In [ ]:
from typing import Annotated, TypedDict, List, Literal
import os
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from IPython.display import Image, display
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
class State(MessagesState):
    """State for the multi-agent system"""
    next_agent: str = ""
    research_data: str = ""
    final_report: str = ""
    task_complete: bool = False
    current_task: str = ""

In [ ]:
llm = ChatAnthropic(model='claude-sonnet-4-6')
llm

In [ ]:
def create_ceo_chain():
    """Determines CEO Delegation Flow"""
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a ceo managing the whole organization:
        
1. Researcher Team - Does the Research and returns their findings
2. Writer Team - Summarizes the findings in clear and concise way
Based on the current state and conversation, decide which team should work next. 
You will pass the work on to the supervisor of that team
If the task is complete, respond with 'DONE'.

Current state:
- Has research data: {has_research}
- Has report: {has_report}

Respond with ONLY the agent name (research_supervisor/writer_supervisor) or 'DONE'.
"""),
        ("human", "{task}")
    ])
    return supervisor_prompt | llm

In [ ]:
def create_research_supervisor_chain():
    """Determines Research Supervisor Delegation Flow"""
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a supervisor managing the research team:
        
1. Data Researcher - Assign to this researcher if data is needed
2. Market Researcher - Assign to this researcher if we need info about the markets
3. Tech Researcher - Assign to this  researcher if we need to learn more about new technologies and R&D
         
Current state:
- Has research data: {has_research}
Respond with ONLY the agent name (data_researcher/market_researcher/tech_researcher) or CEO if done with research.
"""),
        ("human", "{task}")
    ])
    return supervisor_prompt | llm

In [ ]:
def create_writer_supervisor_chain():
    """Determines Writer Supervisor Delegation Flow"""
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a supervisor managing the writer team:
         
        
1. Summary Writer - Creates concise reports that can be read really quickly
2. Long Report Writer - Writes longer reports that follow compliance standards
3. Technical Writer - Writes Technical Reports for Developers
Respond with ONLY the agent name (summary_writer/long_report_writer/technical_writer) or CEO if done with report.
"""),
        ("human", "{task}")
    ])
    return supervisor_prompt | llm

In [ ]:
def ceo_agent(state:State):
    """Anthropic CEO"""
    messages = state["messages"]
    task = messages[-1].content if messages else "No task"

    has_research = bool(state.get("research_data", ""))
    has_report = bool(state.get("final_report", ""))
    chain = create_ceo_chain()
    decision = chain.invoke({
        "has_research":has_research,
        "has_report":has_report,
        "task":task,
    })

    decision_text = decision.content.strip().lower()

    if "researcher_supervisor" in decision_text or not has_research:
        next_agent = "research_supervisor"
        ceo_msg = "CEO: Lets do the research, handing off to Researcher Team..."
    elif "writer_supervisor" in decision_text or (not has_report and has_research):
        next_agent = "writer_supervisor"
        ceo_msg = "CEO: Lets do the report, handing off to Writing Team..."
    else:
        next_agent = "end"        
        ceo_msg = "CEO: All tasks complete! Great work team."
    
    return {
        "messages": [AIMessage(content = ceo_msg)],
        "next_agent": next_agent,
        "current_task": task
    }

In [ ]:
def researcher_supervisor(state:State):
    """Anthropic Research Supervisor"""
    messages = state["messages"]
    task = messages[-1].content if messages else "No task"
    has_research = bool(state.get("research_data", ""))

    chain = create_research_supervisor_chain()
    decision = chain.invoke({
        "task":task,
        "has_research":has_research
    })
    decision_text = decision.content.strip().lower()

    if has_research:
        next_agent = "ceo"        
        supervisor_msg = "Supervisor: All tasks complete! Great work team. Passing Back to CEO..."
    elif "data_researcher" in decision_text:
        next_agent = "data_researcher"
        supervisor_msg = "Supervisor: Lets do with research, handing off to Researcher..."
    elif "market_researcher" in decision_text:
        next_agent = "market_researcher"
        supervisor_msg = "Supervisor: Lets do with analysis, handing off to market researcher..."
    elif "tech_researcher" in decision_text:
        next_agent = "market_researcher"
        supervisor_msg = "Supervisor: Lets do with report, handing off to market researcher..."
    else:
        next_agent = "ceo"        
        supervisor_msg = "Supervisor: All tasks complete! Great work team. Passing Back to CEO..."
    
    return {
        "messages": [AIMessage(content = supervisor_msg)],
        "next_agent": next_agent,
        "current_task": task
    }

In [ ]:
def data_researcher_agent(state:State):
    """Anthropic Data Researcher"""
    task = state.get("current_task", "research_topic")
    research_prompt = f"""As a research specialist, gather data about: {task}

    Include:
    1. Key facts and background
    2. Current trends or developments
    3. Important statistics or data points
    4. Notable examples or case studies
    
    Be concise but thorough."""

    msg = HumanMessage(content=research_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages":f"Researcher: I completed the research on {task}\n\nKey Findings:\n{data[:500]}...\n",
        "next_agent": "research_supervisor",
        "research_data":data
    }

In [ ]:
def market_researcher_agent(state: State):
    """Anthropic Market Researcher"""
    task = state.get("current_task", "research_topic")
    research_prompt = f"""As a market research specialist, analyze market conditions for: {task}

    Include:
    1. Market size and growth trends
    2. Key competitors and market share
    3. Consumer demand and behavior patterns
    4. Market opportunities and risks

    Be concise but thorough."""

    msg = HumanMessage(content=research_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages": f"Market Researcher: I completed the market analysis on {task}\n\nKey Findings:\n{data[:500]}...\n",
        "next_agent": "research_supervisor",
        "research_data": data
    }

In [ ]:
def tech_researcher_agent(state: State):
    """Anthropic Tech Researcher"""
    task = state.get("current_task", "research_topic")
    research_prompt = f"""As a technology research specialist, investigate the tech landscape for: {task}

    Include:
    1. Relevant technologies and tools
    2. Recent R&D breakthroughs or innovations
    3. Technical challenges and limitations
    4. Emerging tech trends and future outlook

    Be concise but thorough."""

    msg = HumanMessage(content=research_prompt)
    content = llm.invoke(input=msg)
    data = content.content

    return {
        "messages": f"Tech Researcher: I completed the technology research on {task}\n\nKey Findings:\n{data[:500]}...\n",
        "next_agent": "research_supervisor",
        "research_data": data
    }

In [ ]:
def writer_supervisor(state: State):
    """Anthropic Writer Supervisor"""
    messages = state["messages"]
    task = messages[-1].content if messages else "No task"
    has_report = bool(state.get("final_report", ""))

    chain = create_writer_supervisor_chain()
    decision = chain.invoke({
        "task": task
    })
    decision_text = decision.content.strip().lower()

    if has_report:
        next_agent = "ceo"
        supervisor_msg = "Writer Supervisor: Report is complete! Passing back to CEO..."
    elif "summary_writer" in decision_text:
        next_agent = "summary_writer"
        supervisor_msg = "Writer Supervisor: Handing off to Summary Writer..."
    elif "long_report_writer" in decision_text:
        next_agent = "long_report_writer"
        supervisor_msg = "Writer Supervisor: Handing off to Long Report Writer..."
    elif "technical_writer" in decision_text:
        next_agent = "technical_writer"
        supervisor_msg = "Writer Supervisor: Handing off to Technical Writer..."
    else:
        next_agent = "ceo"
        supervisor_msg = "Writer Supervisor: Writing complete! Passing back to CEO..."

    return {
        "messages": [AIMessage(content=supervisor_msg)],
        "next_agent": next_agent,
        "current_task": task
    }

In [ ]:
def summary_writer_agent(state: State):
    """Anthropic Summary Writer"""
    task = state.get("current_task", "topic")
    research_data = state.get("research_data", "No research data available.")
    writing_prompt = f"""As a summary writer, create a concise and easy-to-read summary for: {task}

    Research Data:
    {research_data}

    Write a brief summary (3-5 bullet points) that anyone can quickly read and understand.
    Focus on the most important takeaways."""

    msg = HumanMessage(content=writing_prompt)
    content = llm.invoke(input=msg)
    report = content.content

    return {
        "messages": f"Summary Writer: I completed the summary for {task}\n\nSummary:\n{report[:500]}...\n",
        "next_agent": "writer_supervisor",
        "final_report": report
    }

In [ ]:
def long_report_writer_agent(state: State):
    """Anthropic Long Report Writer"""
    task = state.get("current_task", "topic")
    research_data = state.get("research_data", "No research data available.")
    writing_prompt = f"""As a long report writer, create a detailed compliance-ready report for: {task}

    Research Data:
    {research_data}

    Structure the report with:
    1. Executive Summary
    2. Background and Context
    3. Detailed Findings
    4. Analysis and Implications
    5. Recommendations
    6. Conclusion

    Follow formal reporting standards and be thorough."""

    msg = HumanMessage(content=writing_prompt)
    content = llm.invoke(input=msg)
    report = content.content

    return {
        "messages": f"Long Report Writer: I completed the full report for {task}\n\nReport Preview:\n{report[:500]}...\n",
        "next_agent": "writer_supervisor",
        "final_report": report
    }

In [ ]:
def technical_writer_agent(state: State):
    """Anthropic Technical Writer"""
    task = state.get("current_task", "topic")
    research_data = state.get("research_data", "No research data available.")
    writing_prompt = f"""As a technical writer, create a developer-focused technical report for: {task}

    Research Data:
    {research_data}

    Structure the report with:
    1. Technical Overview
    2. Architecture or System Details
    3. Key Technical Specifications
    4. Implementation Considerations
    5. Known Limitations and Trade-offs

    Use precise technical language suitable for a developer audience."""

    msg = HumanMessage(content=writing_prompt)
    content = llm.invoke(input=msg)
    report = content.content

    return {
        "messages": f"Technical Writer: I completed the technical report for {task}\n\nReport Preview:\n{report[:500]}...\n",
        "next_agent": "writer_supervisor",
        "final_report": report
    }

In [ ]:
def router(state:State):
    next_agent = state.get("next_agent", "supervisor")
    if next_agent == "end" or state.get("task_complete", False):
        return END
    elif "supervisor" in next_agent or "writer" in next_agent or "researcher" in next_agent or "ceo" == next_agent:
        return next_agent
    return "supervisor"

In [ ]:
workflow = StateGraph(State)

# --- Nodes ---
workflow.add_node("ceo", ceo_agent)
workflow.add_node("research_supervisor", researcher_supervisor)
workflow.add_node("writer_supervisor", writer_supervisor)
workflow.add_node("data_researcher", data_researcher_agent)
workflow.add_node("market_researcher", market_researcher_agent)
workflow.add_node("tech_researcher", tech_researcher_agent)
workflow.add_node("summary_writer", summary_writer_agent)
workflow.add_node("long_report_writer", long_report_writer_agent)
workflow.add_node("technical_writer", technical_writer_agent)

# --- Entry point ---
workflow.add_edge(START, "ceo")

# --- Shared routing map ---
routing_map = {
    "ceo": "ceo",
    "research_supervisor": "research_supervisor",
    "writer_supervisor": "writer_supervisor",
    "data_researcher": "data_researcher",
    "market_researcher": "market_researcher",
    "tech_researcher": "tech_researcher",
    "summary_writer": "summary_writer",
    "long_report_writer": "long_report_writer",
    "technical_writer": "technical_writer",
    END: END,
}

# --- Conditional edges for every node ---
workflow.add_conditional_edges("ceo", router, routing_map)
workflow.add_conditional_edges("research_supervisor", router, routing_map)
workflow.add_conditional_edges("writer_supervisor", router, routing_map)
workflow.add_conditional_edges("data_researcher", router, routing_map)
workflow.add_conditional_edges("market_researcher", router, routing_map)
workflow.add_conditional_edges("tech_researcher", router, routing_map)
workflow.add_conditional_edges("summary_writer", router, routing_map)
workflow.add_conditional_edges("long_report_writer", router, routing_map)
workflow.add_conditional_edges("technical_writer", router, routing_map)

In [ ]:
# --- Compile ---
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response=graph.invoke(HumanMessage(content="Provide Market Analysis on NVIDIA and Determine if they are worth investing in within the next 5 years"))
response['final_report']